In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from tensorflow import keras
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, models, applications

plt.style.use('default')
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['savefig.facecolor'] = 'white'
plt.rcParams['font.size'] = '20'

In [ ]:
# ============================================
# 1. Data Loading and Preprocessing
# ============================================

def load_data(csv_path, image_dir):
    """
    Loading metadata from a CSV file

    Args:
        csv_path: path to the CSV file
        image_dir: directory with images

    Returns:
        DataFrame with the data
    """
    df = pd.read_csv(csv_path)

    required_cols = ['id', 'pitch_angle', 'filename_0']
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"Missing required column: {col}")

    df['image_path'] = df['filename_0'].apply(lambda x: os.path.join(image_dir, x))

    df['exists'] = df['image_path'].apply(os.path.exists)
    df = df[df['exists']].reset_index(drop=True)

    print(f"Loaded {len(df)} images")
    print(f"Pitch angle range: {df['pitch_angle'].min():.2f} - {df['pitch_angle'].max():.2f} degrees")
    print(f"Mean angle: {df['pitch_angle'].mean():.2f} degrees")
    print(f"Standard deviation: {df['pitch_angle'].std():.2f} degrees")

    return df


def create_train_test_split(df, test_size=0.2, val_size=0.1, random_state=42):
    """
    Splitting data into training, validation, and test sets

    Args:
        df: DataFrame with the data
        test_size: proportion of the test set
        val_size: proportion of the validation set
        random_state: seed for reproducibility

    Returns:
        train_df, val_df, test_df
    """
    train_df, test_df = train_test_split(
        df,
        test_size=test_size,
        random_state=random_state,
        stratify=pd.cut(df['pitch_angle'], bins=10)
    )

    val_ratio = val_size / (1 - test_size)
    train_df, val_df = train_test_split(
        train_df,
        test_size=val_ratio,
        random_state=random_state,
        stratify=pd.cut(train_df['pitch_angle'], bins=10)
    )

    print(f"\nData split:")
    print(f"  Training: {len(train_df)} ({len(train_df) / len(df) * 100:.1f}%)")
    print(f"  Validation: {len(val_df)} ({len(val_df) / len(df) * 100:.1f}%)")
    print(f"  Test: {len(test_df)} ({len(test_df) / len(df) * 100:.1f}%)")

    return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)

In [ ]:
def enhance_spiral_structure(image):
    """
    Enhancing the spiral structure of the image

    Args:
        image: numpy array of the image [H, W, 3] in the range [0, 255] or [0, 1]

    Returns:
        enhanced_image: processed image [H, W, 3] in the range [0, 255]
    """
    if isinstance(image, Image.Image):
        img_array = np.array(image)
    else:
        img_array = image.copy()

    if img_array.max() <= 1.0:
        img_array = (img_array * 255).astype(np.uint8)
    elif img_array.dtype != np.uint8:
        img_array = img_array.astype(np.uint8)

    lab = cv2.cvtColor(img_array, cv2.COLOR_RGB2LAB)
    l_channel = lab[:,:,0].astype(np.float32)

    # ===== 1. Unsharp Masking =====
    blurred = cv2.GaussianBlur(l_channel, (0, 0), sigmaX=1.0, sigmaY=1.0)
    l_sharp = cv2.addWeighted(l_channel, 1.5, blurred, -0.5, 0)

    # ===== 2. CLAHE =====
    clahe = cv2.createCLAHE(clipLimit=1.5, tileGridSize=(8, 8))
    l_enhanced = clahe.apply(l_sharp.astype(np.uint8))

    # ===== 3. FFT High-Pass Filter =====
    f_transform = np.fft.fft2(l_enhanced.astype(np.float32))
    f_shift = np.fft.fftshift(f_transform)

    rows, cols = l_enhanced.shape
    crow, ccol = rows // 2, cols // 2

    mask = np.zeros((rows, cols), np.float32)
    y, x = np.ogrid[:rows, :cols]
    dist_from_center = np.sqrt((x - ccol)**2 + (y - crow)**2)
    mask[(dist_from_center > 5) & (dist_from_center < 60)] = 1.0
    mask = cv2.GaussianBlur(mask, (0, 0), sigmaX=3)

    f_shift_filtered = f_shift * mask
    f_ishift = np.fft.ifftshift(f_shift_filtered)
    l_fft = np.fft.ifft2(f_ishift)
    l_fft = np.abs(l_fft)
    l_fft = cv2.normalize(l_fft, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

    l_combined = cv2.addWeighted(l_enhanced, 0.7, l_fft, 0.3, 0)

    lab_final = lab.copy()
    lab_final[:,:,0] = np.clip(l_combined, 0, 255).astype(np.uint8)
    enhanced_rgb = cv2.cvtColor(lab_final, cv2.COLOR_LAB2RGB)

    return enhanced_rgb.astype(np.float32)

In [ ]:
def deproject_to_face_on(image, target_size=(224, 224)):
    """
    Deprojecting the galaxy to a face-on view with center cropping
    """
    if isinstance(image, Image.Image):
        img_array = np.array(image)
    else:
        img_array = image.copy()

    if img_array.max() <= 1.0:
        img_array = (img_array * 255).astype(np.uint8)
    elif img_array.dtype != np.uint8:
        img_array = img_array.astype(np.uint8)

    if len(img_array.shape) == 3:
        gray = cv2.cvtColor(img_array, cv2.COLOR_RGB2GRAY)
    else:
        gray = img_array.copy()

    h, w = gray.shape

    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    kernel = np.ones((5, 5), np.uint8)
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel, iterations=2)

    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    if len(contours) == 0:
        return cv2.resize(img_array, target_size).astype(np.float32)

    largest_contour = max(contours, key=cv2.contourArea)

    if len(largest_contour) < 5:
        return cv2.resize(img_array, target_size).astype(np.float32)

    ellipse = cv2.fitEllipse(largest_contour)
    (center_x, center_y), (major_axis, minor_axis), angle = ellipse


    if major_axis > 0 and minor_axis > 0:
        ratio = minor_axis / major_axis
        ratio = np.clip(ratio, 0.0, 1.0)
        inclination = np.arccos(ratio) * 180 / np.pi
        scale_y = major_axis / minor_axis
    else:
        inclination = 0
        scale_y = 1.0

    print(f"Deprojection:")
    print(f"  Center: ({center_x:.1f}, {center_y:.1f})")
    print(f"  Axes: {major_axis:.1f} / {minor_axis:.1f}")
    print(f"  Inclination: {inclination:.1f}°")
    print(f"  Scale Y: {scale_y:.2f}")

    # ===== 1. DEPROJECTION =====
    cx, cy = center_x, center_y

    T1 = np.array([[1, 0, -cx], [0, 1, -cy], [0, 0, 1]], dtype=np.float32)
    angle_rad = -np.deg2rad(angle)
    R1 = np.array([
        [np.cos(angle_rad), -np.sin(angle_rad), 0],
        [np.sin(angle_rad), np.cos(angle_rad), 0],
        [0, 0, 1]
    ], dtype=np.float32)
    S = np.array([[1, 0, 0], [0, scale_y, 0], [0, 0, 1]], dtype=np.float32)
    angle_rad_back = np.deg2rad(angle)
    R2 = np.array([
        [np.cos(angle_rad_back), -np.sin(angle_rad_back), 0],
        [np.sin(angle_rad_back), np.cos(angle_rad_back), 0],
        [0, 0, 1]
    ], dtype=np.float32)
    T2 = np.array([[1, 0, cx], [0, 1, cy], [0, 0, 1]], dtype=np.float32)

    M = T2 @ R2 @ S @ R1 @ T1

    corners = np.array([[0, 0, 1], [w, 0, 1], [0, h, 1], [w, h, 1]], dtype=np.float32)
    transformed_corners = (M @ corners.T).T
    x_min, y_min = transformed_corners[:, :2].min(axis=0)
    x_max, y_max = transformed_corners[:, :2].max(axis=0)

    new_w = int(np.ceil(x_max - x_min))
    new_h = int(np.ceil(y_max - y_min))

    M_adjusted = M.copy()
    M_adjusted[0, 2] -= x_min
    M_adjusted[1, 2] -= y_min

    deprojected = cv2.warpAffine(
        img_array,
        M_adjusted[:2, :],
        (new_w, new_h),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0
    )

    # ===== 2. CENTER CROPPING =====
    crop_size = int(major_axis * 1.5)

    new_cx = center_x - x_min
    new_cy = center_y - y_min

    x1 = max(0, int(new_cx - crop_size // 2))
    y1 = max(0, int(new_cy - crop_size // 2))
    x2 = min(new_w, int(new_cx + crop_size // 2))
    y2 = min(new_h, int(new_cy + crop_size // 2))

    cropped = deprojected[y1:y2, x1:x2]

    print(f"Cropping: ({x1}, {y1}) - ({x2}, {y2})")
    print(f"Size after cropping: {cropped.shape[1]} x {cropped.shape[0]}")

    # ===== 3. RESIZE TO TARGET SIZE =====
    final_image = cv2.resize(cropped, target_size, interpolation=cv2.INTER_LINEAR)

    print(f"Final size: {target_size[0]} x {target_size[1]}")
    print(f"Deprojection completed")

    return final_image.astype(np.float32)

In [ ]:
# ============================================
# 2. Creating the Data Pipeline
# ============================================

class GalaxyDataGenerator(keras.utils.Sequence):
    """
    Data generator for loading 600x600 galaxy images
    """

    def __init__(self, df, image_dir, batch_size=32,
                 input_size=(224, 224),
                 original_size=(600, 600),
                 augment=False,
                 shuffle=True,
                 enhance_spirals=False):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.batch_size = batch_size
        self.input_size = input_size
        self.original_size = original_size
        self.augment = augment
        self.shuffle = shuffle
        self.enhance_spirals = enhance_spirals

        if self.augment:
            self.augmentation_layer = keras.Sequential([
                layers.RandomFlip("horizontal"),
                layers.RandomFlip("vertical"),
                layers.RandomRotation(0.15),
                layers.RandomZoom(0.15),
                layers.RandomContrast(0.15),
                layers.RandomBrightness(0.1),
            ], name="augmentation")

        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.df) / self.batch_size))

    def __getitem__(self, index):
        start_idx = index * self.batch_size
        end_idx = min((index + 1) * self.batch_size, len(self.df))
        batch_indices = list(range(start_idx, end_idx))

        actual_batch_size = len(batch_indices)
        images = np.zeros((actual_batch_size, *self.input_size, 3), dtype=np.float32)
        labels = np.zeros((actual_batch_size, 1), dtype=np.float32)

        for i, idx in enumerate(batch_indices):
            row = self.df.iloc[idx]
            img_path = os.path.join(self.image_dir, row['filename_0'])

            try:
                img = keras.utils.load_img(
                    img_path,
                    target_size=self.input_size,
                    color_mode='rgb'
                )

                img_array = keras.utils.img_to_array(img)

                if self.enhance_spirals:
                    img_array = enhance_spiral_structure(img_array)

                if img_array.mean() < 1:
                    print(f"Warning: Empty image {img_path}")

                images[i] = img_array
                labels[i] = row['pitch_angle']

            except Exception as e:
                print(f"Error loading {img_path}: {e}")
                images[i] = np.zeros((*self.input_size, 3), dtype=np.float32)
                labels[i] = 0.0

        if self.augment:
            images = self.augmentation_layer(images, training=True)

        return images, labels

    def on_epoch_end(self):
        if self.shuffle:
            self.df = self.df.sample(frac=1, random_state=42).reset_index(drop=True)


def create_data_generators(train_df, val_df, test_df, image_dir, batch_size=32,
                           input_size=(224, 224), original_size=(600, 600)):
    """
    Creating data generators for training, validation, and testing

    Args:
        train_df: DataFrame with training data
        val_df: DataFrame with validation data
        test_df: DataFrame with test data
        image_dir: directory with images
        batch_size: batch size
        input_size: image size to feed into the model (224x224 for MobileNet)
        original_size: original image size (600x600)

    Returns:
        train_gen, val_gen, test_gen
    """
    train_gen = GalaxyDataGenerator(
        train_df, image_dir, batch_size=batch_size,
        input_size=input_size,
        original_size=original_size,
        augment=True, shuffle=True,
        enhance_spirals=True
    )

    val_gen = GalaxyDataGenerator(
        val_df, image_dir, batch_size=batch_size,
        input_size=input_size,
        original_size=original_size,
        augment=False, shuffle=False,
        enhance_spirals=True
    )

    test_gen = GalaxyDataGenerator(
        test_df, image_dir, batch_size=batch_size,
        input_size=input_size,
        original_size=original_size,
        augment=False, shuffle=False,
        enhance_spirals=True
    )

    return train_gen, val_gen, test_gen

In [ ]:
# ============================================
# 3. Building the MobileNet Model for Regression
# ============================================

def create_mobilenet_regression_model(input_shape=(224, 224, 3),
                                      pretrained_weights='imagenet',
                                      dropout_rate=0.4,
                                      learning_rate=1e-4):
    """
    Creating a MobileNetV2 model for a regression task
    """
    base_model = applications.MobileNetV2(
        input_shape=input_shape,
        include_top=False,
        weights=pretrained_weights,
        pooling='avg',
        alpha=1.0
    )

    base_model.trainable = False

    inputs = keras.Input(shape=input_shape)

    x = applications.mobilenet_v2.preprocess_input(inputs)

    x = base_model(x, training=False)

    x = layers.Dense(512, activation='relu',
                     kernel_regularizer=keras.regularizers.l2(0.001))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout_rate)(x)

    x = layers.Dense(256, activation='relu',
                     kernel_regularizer=keras.regularizers.l2(0.001))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout_rate / 2)(x)

    x = layers.Dense(128, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout_rate / 4)(x)

    outputs = layers.Dense(1, activation='linear')(x)

    model = models.Model(inputs, outputs, name='mobilenet_pitch_angle_regression')

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='mse',
        metrics=[
            'mae',
            keras.metrics.RootMeanSquaredError(name='rmse'),
            keras.metrics.MeanAbsolutePercentageError(name='mape')
        ]
    )

    print("\nModel structure:")
    model.summary()

    return model


def unfreeze_top_layers(model, unfreeze_layers=20):
    """
    Unfreezing the top layers of the base model for fine-tuning

    Args:
        model: Keras model
        unfreeze_layers: number of layers to unfreeze

    Returns:
        model with unfrozen layers
    """
    base_model = model.get_layer('mobilenetv2_1.00_224')

    for layer in base_model.layers[:-unfreeze_layers]:
        layer.trainable = False
    for layer in base_model.layers[-unfreeze_layers:]:
        layer.trainable = True

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-5),
        loss='mse',
        metrics=['mae', keras.metrics.RootMeanSquaredError(name='rmse')]
    )

    print(f"\nUnfroze {unfreeze_layers} layers of the base model")

    return model

In [ ]:
# ============================================
# 4. Model Training
# ============================================

def train_model(model, train_gen, val_gen, epochs=50, callbacks_list=None):
    """
    Training the model

    Args:
        model: Keras model
        train_gen: training data generator
        val_gen: validation data generator
        epochs: number of epochs
        callbacks_list: list of callbacks

    Returns:
        training history
    """

    if callbacks_list is None:
        callbacks_list = [
            keras.callbacks.EarlyStopping(
                monitor='val_loss',
                patience=10,
                restore_best_weights=True,
                verbose=1
            ),
            keras.callbacks.ReduceLROnPlateau(
                monitor='val_loss',
                factor=0.5,
                patience=5,
                min_lr=1e-7,
                verbose=1
            ),
            keras.callbacks.ModelCheckpoint(
                'best_pitch_angle_model.keras',
                monitor='val_loss',
                save_best_only=True,
                verbose=1
            ),
            keras.callbacks.CSVLogger('training_history.csv')
        ]

    history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=epochs,
        callbacks=callbacks_list,
        verbose=1
    )

    return history

In [ ]:
# ============================================
# 5. Model Evaluation
# ============================================

def evaluate_model(model, test_gen, test_df):
    """
    Evaluating the model on the test set

    Args:
        model: trained model
        test_gen: test data generator
        test_df: DataFrame with test data

    Returns:
        metrics and predictions
    """

    # Evaluation
    results = model.evaluate(test_gen, verbose=1)

    print("\n" + "=" * 50)
    print("Results on the test set:")
    print("=" * 50)
    print(f"  MSE:  {results[0]:.4f}")
    print(f"  MAE:  {results[1]:.4f} degrees")
    print(f"  RMSE: {results[2]:.4f} degrees")

    predictions = model.predict(test_gen)
    true_values = test_df['pitch_angle'].values

    errors = np.abs(predictions.flatten() - true_values)
    print(f"\nError statistics:")
    print(f"  Minimum error: {errors.min():.2f} degrees")
    print(f"  Maximum error: {errors.max():.2f} degrees")
    print(f"  Median error: {np.median(errors):.2f} degrees")
    print(f"  95th percentile: {np.percentile(errors, 95):.2f} degrees")

    return results, predictions, true_values

In [ ]:
# ============================================
# 6. Results Visualization
# ============================================

def plot_training_history(history):
    """
    Visualizing the training history
    """
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].plot(history.history['loss'], label='Train Loss')
    axes[0].plot(history.history['val_loss'], label='Val Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('MSE Loss')
    axes[0].set_title('Loss History')
    axes[0].legend()
    axes[0].grid(True)

    axes[1].plot(history.history['mae'], label='Train MAE')
    axes[1].plot(history.history['val_mae'], label='Val MAE')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('MAE (degrees)')
    axes[1].set_title('MAE History')
    axes[1].legend()
    axes[1].grid(True)

    if 'lr' in history.history:
        axes[2].plot(history.history['lr'])
        axes[2].set_xlabel('Epoch')
        axes[2].set_ylabel('Learning Rate')
        axes[2].set_title('Learning Rate Schedule')
        axes[2].grid(True)

    plt.tight_layout()
    plt.savefig('training_history.png', dpi=150)
    plt.show()


def plot_predictions_vs_true(true_values, predictions):
    """
    Plot of predictions vs true values
    """
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    axes[0].scatter(true_values, predictions.flatten(), alpha=0.5, s=50)
    axes[0].plot([true_values.min(), true_values.max()],
                 [true_values.min(), true_values.max()],
                 'r--', linewidth=2, label='Ideal')
    axes[0].set_xlabel('True Pitch Angle (degrees)')
    axes[0].set_ylabel('Predicted Pitch Angle (degrees)')
    axes[0].set_title('Predictions vs True Values')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    errors = predictions.flatten() - true_values
    axes[1].hist(errors, bins=50, edgecolor='black', alpha=0.7)
    axes[1].axvline(x=0, color='r', linestyle='--', linewidth=2)
    axes[1].set_xlabel('Error (degrees)')
    axes[1].set_ylabel('Frequency')
    axes[1].set_title('Error Distribution')
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('predictions_analysis.png', dpi=150)
    plt.show()

In [ ]:
# ============================================
# 7. Main Training Script
# ============================================

def main():
    """
    Main script for training the model
    """

    CSV_PATH = './screenshots/metadata.csv'
    IMAGE_DIR = './screenshots'

    ORIGINAL_IMAGE_SIZE = (600, 600)
    INPUT_IMAGE_SIZE = (224, 224)

    BATCH_SIZE = 32
    EPOCHS_PHASE1 = 20
    EPOCHS_PHASE2 = 20

    print("=" * 60)
    print("Training the model to predict the pitch angle of spiral arms")
    print(f"Original image size: {ORIGINAL_IMAGE_SIZE}")
    print(f"Model input size: {INPUT_IMAGE_SIZE}")
    print("=" * 60)

    print("\n[1/7] Loading data...")
    df = load_data(CSV_PATH, IMAGE_DIR)

    print("\n[2/7] Splitting into training/validation/test sets...")
    train_df, val_df, test_df = create_train_test_split(df, test_size=0.2, val_size=0.1)

    print("\n[3/7] Creating data generators...")
    train_gen, val_gen, test_gen = create_data_generators(
        train_df, val_df, test_df, IMAGE_DIR,
        batch_size=BATCH_SIZE,
        input_size=INPUT_IMAGE_SIZE,
        original_size=ORIGINAL_IMAGE_SIZE
    )

    print("\n[4/7] Creating the MobileNetV2 model...")
    model = create_mobilenet_regression_model(
        input_shape=(*INPUT_IMAGE_SIZE, 3),
        pretrained_weights='imagenet',
        dropout_rate=0.4,
        learning_rate=1e-4
    )

    print("\n[5/7] Training the model (Phase 1 - frozen base)...")
    history_phase1 = train_model(
        model, train_gen, val_gen,
        epochs=EPOCHS_PHASE1,
        callbacks_list=[
            keras.callbacks.EarlyStopping(
                monitor='val_loss', patience=15,
                restore_best_weights=True, verbose=1
            ),
            keras.callbacks.ReduceLROnPlateau(
                monitor='val_loss', factor=0.5,
                patience=7, min_lr=1e-7, verbose=1
            ),
            keras.callbacks.ModelCheckpoint(
                'best_model_phase1.keras', monitor='val_loss',
                save_best_only=True, verbose=1
            ),
            keras.callbacks.CSVLogger('training_history_phase1.csv')
        ]
    )

    print("\n[6/7] Fine-tuning the model (Phase 2 - unfrozen base)...")
    model = unfreeze_top_layers(model, unfreeze_layers=30)
    history_phase2 = train_model(
        model, train_gen, val_gen,
        epochs=EPOCHS_PHASE2,
        callbacks_list=[
            keras.callbacks.EarlyStopping(
                monitor='val_loss', patience=15,
                restore_best_weights=True, verbose=1
            ),
            keras.callbacks.ReduceLROnPlateau(
                monitor='val_loss', factor=0.5,
                patience=7, min_lr=1e-7, verbose=1
            ),
            keras.callbacks.ModelCheckpoint(
                'best_model_phase2.keras', monitor='val_loss',
                save_best_only=True, verbose=1
            ),
            keras.callbacks.CSVLogger('training_history_phase2.csv')
        ]
    )

    print("\n[7/7] Final evaluation on the test set")
    print("=" * 60)
    results, predictions, true_values = evaluate_model(model, test_gen, test_df)

    model.save('pitch_angle_regression_model_600x600.keras')
    print("\nModel saved to 'pitch_angle_regression_model_600x600.keras'")

    config = {
        'original_image_size': ORIGINAL_IMAGE_SIZE,
        'input_image_size': INPUT_IMAGE_SIZE,
        'batch_size': BATCH_SIZE,
        'epochs_phase1': EPOCHS_PHASE1,
        'epochs_phase2': EPOCHS_PHASE2,
        'num_train_samples': len(train_df),
        'num_val_samples': len(val_df),
        'num_test_samples': len(test_df),
        'pitch_angle_range': [float(df['pitch_angle'].min()),
                              float(df['pitch_angle'].max())]
    }

    import json
    with open('model_config.json', 'w') as f:
        json.dump(config, f, indent=2)
    print("Configuration saved to 'model_config.json'")

    print("\n" + "=" * 60)
    print("Training completed successfully!")
    print("=" * 60)

    return model, history_phase1, history_phase2

In [ ]:
# ============================================
# 8. Function for Inference (Prediction)
# ============================================

def predict_pitch_angle(model, image_path,
                        original_size=(600, 600),
                        input_size=(224, 224),
                        enhance_spirals=False):
    """
    Predicting the pitch angle for a single 600x600 image

    Args:
        model: trained model
        image_path: path to the image
        original_size: original image size
        input_size: size for feeding into the model

    Returns:
        predicted angle in degrees
    """
    img = keras.utils.load_img(image_path, target_size=original_size)
    img = img.resize(input_size)

    img_array = keras.utils.img_to_array(img)

    if enhance_spirals:
        img_array = enhance_spiral_structure(img_array)
        img_deproject = deproject_to_face_on(img_array)
        img_pil = keras.utils.array_to_img(img_deproject)
        img_pil = img_pil.resize(input_size)
        img_array = keras.utils.img_to_array(img_pil)

    img_array = np.expand_dims(img_array, axis=0)

    prediction = model.predict(img_array, verbose=0)

    return float(prediction[0, 0])

In [ ]:
model, history1, history2 = main()

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image


import matplotlib.ticker as ticker

def test_folder_with_labels(data_dir, model, max_images=32, enhance_spirals = False):
    """
    Testing a folder with images where labels are in the filename
    Filename format: 'anything;label.ext' or 'anything_label.ext'
    """
    if not os.path.exists(data_dir):
        print(f"Folder not found: {data_dir}")
        return

    true_labels = []
    predicted_labels = []
    image_paths = []

    files = [f for f in os.listdir(data_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    files = files[:max_images]

    print(f"\n{'=' * 60}")
    print(f"Testing {len(files)} images from {data_dir}")
    print(f"{'=' * 60}\n")

    n_cols = 4
    n_rows = (len(files) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 5 * n_rows))
    axes = axes.flatten() if n_rows * n_cols > 1 else [axes]

    for i, filename in enumerate(files):
        image_path = os.path.join(data_dir, filename)

        try:
            label = None
            if ';' in filename:
                text = filename[filename.find(';') + 1:]
                label = float(text.split('.')[0] + '.' + text.split('.')[1])
            elif ' ' in filename:
                parts = filename.replace('.png', '').replace('.jpg', '').split(' ')
                try:
                    label = float(parts[-1])
                except:
                    pass

            img = Image.open(image_path)

            predicted_label = predict_pitch_angle(model, image_path, enhance_spirals=enhance_spirals)

            if label is not None:
                true_labels.append(label)
            predicted_labels.append(predicted_label)
            image_paths.append(filename)

            axes[i].imshow(img)
            if label is not None:
                title_text = f"File: {filename[:20]}...\nTrue: {label:.2f}°\nPrediction: {predicted_label:.2f}°"
                axes[i].set_title(title_text.replace('.', ','), fontsize=8) # <-- Replaced
            else:
                title_text = f"Prediction: {predicted_label:.2f}°"
                axes[i].set_title(title_text.replace('.', ','), fontsize=8) # <-- Replaced
            axes[i].axis('off')

        except Exception as e:
            print(f"Error processing {filename}: {e}")
            axes[i].axis('off')

    for j in range(len(files), len(axes)):
        axes[j].axis('off')

    plt.tight_layout()
    plt.savefig('test_results.png', dpi=150, bbox_inches='tight')
    print("Visualization saved to 'test_results.png'")
    plt.show()

    if true_labels:
        true_labels = np.array(true_labels)
        predicted_labels = np.array(predicted_labels)

        errors = np.abs(predicted_labels - true_labels)

        print(f"\n{'=' * 60}")
        print("QUALITY METRICS")
        print(f"{'=' * 60}")
        print(f"Number of images: {len(true_labels)}")

        # REPLACING DOT WITH COMMA IN METRICS OUTPUT
        print(f"Mean error (MAE): {errors.mean():.2f}°".replace('.', ','))
        print(f"Median error: {np.median(errors):.2f}°".replace('.', ','))
        print(f"Std. deviation of error: {errors.std():.2f}°".replace('.', ','))
        print(f"Min error: {errors.min():.2f}°".replace('.', ','))
        print(f"Max error: {errors.max():.2f}°".replace('.', ','))
        print(f"95th percentile: {np.percentile(errors, 95):.2f}°".replace('.', ','))

        plt.figure(figsize=(12, 5))

        fmt = ticker.FuncFormatter(lambda x, pos: f'{x:.1f}'.replace('.', ','))

        plt.subplot(1, 2, 1)
        plt.scatter(true_labels, predicted_labels, alpha=0.6, s=50)
        plt.plot([true_labels.min(), true_labels.max()],
                 [true_labels.min(), true_labels.max()], 'r--', linewidth=2)
        plt.xlabel('True Angle (°)')
        plt.ylabel('Predicted Angle (°)')
        plt.title('Predictions vs True')
        plt.grid(True, alpha=0.3)

        plt.gca().xaxis.set_major_formatter(fmt)
        plt.gca().yaxis.set_major_formatter(fmt)

        plt.subplot(1, 2, 2)
        plt.hist(errors, bins=20, edgecolor='black', alpha=0.7)

        mean_err_str = f'Mean: {errors.mean():.2f}°'.replace('.', ',')
        plt.axvline(errors.mean(), color='r', linestyle='--', label=mean_err_str)

        plt.xlabel('Error (°)')
        plt.ylabel('Count')
        plt.title('Error Distribution')
        plt.legend()
        plt.grid(True, alpha=0.3)

        plt.gca().xaxis.set_major_formatter(fmt)

        plt.tight_layout()
        plt.savefig('error_analysis.png', dpi=150, bbox_inches='tight')
        print("Error analysis saved to 'error_analysis.png'")
        plt.show()

In [ ]:
test_folder_with_labels('./test_real_data_crop', model, max_images=65, enhance_spirals = True)

In [ ]:
model = create_mobilenet_regression_model()
model.load_weights('./models/with_preprocess/pitch_angle_regression_model_600x600.h5')